# Day 041 — AdaBoost Deep Dive: Weights, Math & Code

> **Approach:** Every concept explained plainly first, then the math, then running code to see the result.  
> **No prior knowledge required** beyond basic Python.

---

## Part 1 — What is AdaBoost? (Analogy)

Imagine you're preparing for a competitive exam:

1. You take a mock test — **some questions are wrong**
2. Your teacher gives you **more practice on those wrong questions** (gives them more weight)
3. You take another test — again some wrong — teacher **focuses even harder on those**
4. This process repeats over and over
5. In the end, all your attempts are combined → **much stronger performance**

That's exactly **AdaBoost** (Adaptive Boosting):
- Train a weak model first
- **Increase the weight** of misclassified samples
- The next model pays more attention to those hard samples
- Combine all models with a weighted vote → one strong final model

---

## Part 2 — Setup: Dataset & Imports

We'll use the **Breast Cancer dataset** (sklearn built-in):
- **569 samples**, **30 features**
- Binary classification: malignant (0) or benign (1)

The base estimator will be a **Decision Stump** (`max_depth=1`) — a tree with just one rule.

In [1]:
from sklearn.tree import DecisionTreeClassifier

In [2]:
stamp=DecisionTreeClassifier(max_depth=1)

In [3]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [4]:
X,y=load_breast_cancer(return_X_y=True)

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## Part 3 — Building the AdaBoost Model

Key parameters of `AdaBoostClassifier`:

| Parameter | Meaning | Our value |
|-----------|---------|----------|
| `estimator` | The base weak learner — here a Decision Stump | `DecisionTreeClassifier(max_depth=1)` |
| `n_estimators` | How many stumps to train | `50` |
| `learning_rate` | How much each stump's contribution is scaled | `1.0` |
| `random_state` | For reproducibility | `42` |

> A **Decision Stump** is an extremely weak classifier — it only makes one split on one feature.  
> AdaBoost combines 50 such weak stumps to build one strong classifier!

In [6]:
ada=AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),n_estimators=50,learning_rate=1.0,random_state=42)

## Part 4 — Training the Model

When you call `ada.fit()`, AdaBoost does the following internally:

**Iteration 1:**
- Give all samples **equal weight**: $w_i = \frac{1}{n}$
- Train a Decision Stump
- Increase the weight of misclassified samples

**Iterations 2, 3, ... 50:**
- The next stump **pays more attention** to samples with higher weights
- Weights are updated again after each round

**Final Prediction:**
$$F(x) = \text{sign}\left(\sum_{m=1}^{M} \alpha_m \cdot h_m(x)\right)$$

Where $\alpha_m$ = the **voting power** of stump $m$, determined by its accuracy.

In [7]:
ada.fit(X_train,y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   random_state=42)

## Part 5 — Prediction & Accuracy

Now let's make predictions on the test set and check accuracy.

In [8]:
y_pred=ada.predict(X_test)

In [10]:
print(accuracy_score(y_test,y_pred))

0.9649122807017544


**96.49% accuracy!** And we only used decision stumps — some of the weakest possible classifiers.

---

## Part 6 — Estimator Weights (`estimator_weights_`)

Each stump's **voting power** ($\alpha_m$) is determined by how accurate it was:

$$\alpha_m = \frac{1}{2} \ln\left(\frac{1 - \epsilon_m}{\epsilon_m}\right)$$

Where $\epsilon_m$ = the weighted error rate (between 0 and 0.5).

- If a stump classifies **very well** → $\epsilon_m$ is small → $\alpha_m$ is **large** (more voting power)
- If a stump performs **poorly** → $\alpha_m$ is small or even negative

> **Expect:** Early iterations have higher weights (easier samples),  
> later ones have lower weights (harder re-weighted samples).

In [11]:
ada.estimator_weights_

array([2.45435198, 1.85396203, 1.64912402, 1.13265675, 1.17354849,
       0.7402008 , 0.87339663, 1.2190775 , 0.67400709, 1.02630105,
       0.62857029, 1.0652562 , 0.84662335, 0.88364558, 0.89724546,
       0.84138712, 0.49458534, 0.88326823, 0.94105224, 0.89117048,
       0.51708815, 0.78989262, 0.48830505, 0.56584274, 0.79766429,
       0.580027  , 0.81079972, 0.75265042, 0.77863326, 0.33381544,
       0.80730587, 0.74721269, 0.31100309, 0.8383608 , 0.55808476,
       0.60566564, 0.57489299, 0.28055258, 0.35398844, 0.61724206,
       0.71257471, 0.71355604, 0.84895329, 0.48119548, 0.90138034,
       0.69940088, 0.81875494, 0.4015193 , 0.35367521, 0.82038696])

**Observation:** The first stump (weight ≈ 2.45) was the most confident — it trained on easy, equally-weighted samples.  
Later stumps generally have lower weights because they struggled on harder, re-weighted samples.

---

## Part 7 — Dataset Shape

A quick sanity check on our data dimensions.

In [12]:
X.shape

(569, 30)

**569 samples, 30 features** — confirmed.

---

## Part 8 — Estimator Errors (`estimator_errors_`)

This is the **weighted error rate** ($\epsilon_m$) for each stump:

$$\epsilon_m = \sum_{i=1}^{n} w_i \cdot \mathbb{1}[h_m(x_i) \neq y_i]$$

i.e., the sum of weights of all misclassified samples at that round.

- **Iteration 1:** Error is very small (≈ 0.079) — stumps classify easily on uniform weights
- **Later iterations:** Error grows (≈ 0.4) — samples become harder as misclassified ones get more weight

> **Key Insight:** The error grows because AdaBoost keeps boosting the weight of misclassified samples,  
> making the task progressively harder for each subsequent stump.

In [13]:
ada.estimator_errors_

array([0.07912088, 0.13540838, 0.16122738, 0.24367114, 0.23621418,
       0.32296023, 0.29454803, 0.22809883, 0.33760017, 0.26380185,
       0.34783479, 0.25630627, 0.30014167, 0.29242289, 0.28961688,
       0.30124272, 0.37881398, 0.29250098, 0.28068784, 0.29086834,
       0.37353337, 0.31219173, 0.38029294, 0.36219664, 0.31052537,
       0.35892638, 0.30772011, 0.32024406, 0.31461453, 0.41731256,
       0.3084649 , 0.32142895, 0.42286991, 0.30188013, 0.36399073,
       0.35304856, 0.36010856, 0.43031831, 0.41241557, 0.35040896,
       0.32903017, 0.32881356, 0.29965248, 0.38196987, 0.28876692,
       0.33194507, 0.30602801, 0.40094737, 0.41249148, 0.30568153])

**Pattern clearly visible:** Error started at 0.079 and grew to ~0.3–0.4.  
This confirms that AdaBoost's weight-update mechanism is working exactly as designed.

---

## Part 9 — Effect of Learning Rate

The **learning rate** (`η`) controls how much each stump contributes to the final prediction:

$$F(x) = \text{sign}\left(\sum_{m=1}^{M} \eta \cdot \alpha_m \cdot h_m(x)\right)$$

- **Small `η` (e.g., 0.1):** Each stump contributes a little → slower learning, needs more estimators
- **Large `η` (e.g., 1.0):** Each stump contributes fully → faster learning

> **Trade-off:** Learning rate and `n_estimators` have an inverse relationship.  
> Smaller learning rate + more estimators ≈ larger learning rate + fewer estimators (roughly)

Let's compare accuracy across different learning rates:

In [14]:
for lr in [0.1, 0.5, 1.0, 1.5]:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=50,
        learning_rate=lr,
        random_state=42
    )
    ada.fit(X_train, y_train)
    acc = accuracy_score(y_test, ada.predict(X_test))
    print(f"learning_rate={lr} -> accuracy={acc:.4f}")

learning_rate=0.1 -> accuracy=0.9561
learning_rate=0.5 -> accuracy=0.9649
learning_rate=1.0 -> accuracy=0.9649
learning_rate=1.5 -> accuracy=0.9649


## Part 10 — Results Analysis

| Learning Rate | Accuracy | Observation |
|---------------|----------|-------------|
| 0.1 | 95.61% | Too slow — under-converged with only 50 estimators |
| 0.5 | 96.49% | Good balance |
| 1.0 | 96.49% | Default — optimal here |
| 1.5 | 96.49% | Same result — model has plateaued on this dataset |

### Key Takeaways:

1. **`lr=0.1`** gives slightly lower accuracy because the model hasn't fully converged in 50 steps. With `n_estimators=500` it would likely match 96.49%.

2. **`lr >= 0.5`** all produce the same accuracy — 50 estimators are sufficient for full convergence on this dataset.

3. **The power of boosting:** ~96.5% accuracy using only decision stumps — some of the weakest possible learners.

---

## Summary — Full AdaBoost Algorithm Flow

```
START
  |
  ├── Initialize equal weights for all samples: w_i = 1/n
  |
  └── For m = 1 to M:
        |
        ├── Train a weighted decision stump h_m(x)
        |
        ├── Compute weighted error:  ε_m = Σ w_i · 1[h_m(x_i) ≠ y_i]
        |
        ├── Compute stump voting power:  α_m = 0.5 · ln((1 - ε_m) / ε_m)
        |
        └── Update sample weights:
              - Misclassified: w_i ← w_i · exp(+α_m)  [increase]
              - Correct:       w_i ← w_i · exp(-α_m)  [decrease]
              - Normalize:     Σ w_i = 1

Final Prediction: F(x) = sign(Σ α_m · h_m(x))
```

> **AdaBoost vs Gradient Boosting:**  
> - AdaBoost adjusts **sample weights** to focus on hard examples  
> - Gradient Boosting fits on **residuals** (see Day 040!)